In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Load the SAVED fine-tuned model instead of retraining

Mounted at /content/drive


In [ ]:
# 2. Load the SAVED fine-tuned model instead of retraining
from transformers import BartTokenizer, BartForConditionalGeneration
import torch, json

save_path = '/content/drive/MyDrive/meeting_summarizer/bart_finetuned_final'
tokenizer = BartTokenizer.from_pretrained(save_path)
model = BartForConditionalGeneration.from_pretrained(save_path)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print("Loaded fine-tuned model from", save_path)



[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loaded fine-tuned model from /content/drive/MyDrive/meeting_summarizer/bart_finetuned_final


In [ ]:
import json

def flatten_transcript_json(data, mode="named", merge_consecutive=True):   # <- default changed here
    if isinstance(data, str):
        data = json.loads(data)
    turns = data.get("transcript", data) if isinstance(data, dict) else data

    speaker_order = {}
    def get_tag(turn):
        raw_speaker = turn.get("speaker", "Unknown")
        if mode == "match_training":
            if raw_speaker not in speaker_order:
                speaker_order[raw_speaker] = f"S{len(speaker_order) + 1}"
            return speaker_order[raw_speaker]

        # named mode
        name = turn.get("identified_name")
        role = turn.get("role")
        has_real_name = bool(name) and name != raw_speaker
        if has_real_name and role:
            return f"{name} ({role})"
        elif has_real_name:
            return name
        elif role:
            return f"{raw_speaker} ({role})"
        else:
            return raw_speaker

    flat_turns = []
    for turn in turns:
        text = (turn.get("text") or "").strip()
        if not text:
            continue
        flat_turns.append((get_tag(turn), text))

    if merge_consecutive:
        merged = []
        for tag, text in flat_turns:
            if merged and merged[-1][0] == tag:
                merged[-1] = (tag, merged[-1][1] + " " + text)
            else:
                merged.append((tag, text))
        flat_turns = merged

    return "  ".join(f"{tag}: {text}" for tag, text in flat_turns)


def summarize_transcript_v2(transcript, mode="named"):   # <- default changed here too
    if isinstance(transcript, (dict, list)):
        transcript = flatten_transcript_json(transcript, mode=mode)
    elif isinstance(transcript, str):
        stripped = transcript.strip()
        if stripped.startswith("{") or stripped.startswith("["):
            transcript = flatten_transcript_json(transcript, mode=mode)
    return summarize_transcript(transcript)

In [ ]:
# 3. Same generation functions as cell 15 (unchanged) — but WITHOUT
#    the trailing "Test it" block, since test_df won't exist here
def summarize_transcript(transcript: str) -> str:
    words = transcript.split()
    if len(words) <= 700:
        return _generate(transcript)
    chunk_size, overlap, chunks, start = 600, 100, [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(' '.join(words[start:end]))
        start += chunk_size - overlap
        if end == len(words):
            break
    partial_summaries = [_generate(c) for c in chunks]
    if len(partial_summaries) == 1:
        return partial_summaries[0]
    return _generate(" ".join(partial_summaries))

def _generate(text: str) -> str:
    input_text = 'summarize meeting: ' + text
    inputs = tokenizer(input_text, return_tensors='pt', max_length=1024, truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=512, num_beams=4,
                              length_penalty=2.0, early_stopping=True, no_repeat_ngram_size=3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. Same adapter as cell 16 (unchanged) — paste flatten_transcript_json() and summarize_transcript_v2() here

# 5. Use it on whatever new Step-6 JSON you have
with open('/content/drive/MyDrive/meeting_summarizer/step6_input_0001.json') as f:
    step6_data = json.load(f)

summary = summarize_transcript_v2(step6_data)
print(summary)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

The meeting was the kick-off meeting for a new remote control design project for Planet Neos. The project brief included three stages: original, trendy and user-friendly design, user interface, and industrial design. Andrew presented a PowerPoint presentation for the project and asked the team to create a whiteboard sketch of their favorite animal and describe their favorite characteristics. The functional design stage is the first stage of the design process, followed by technical design. The next meeting would be in 30 minutes and the industrial designer would work on the actual working design of the remote.
